# 第 5 天挑战 —— 公司专家系统（Knowledge Base Q&A）

## 练习目标

官方 Day 5 常见做法是「抓网页 → 写宣传册」。这里换了玩法：为一个公司搭**专家系统**：

1. 输入公司名 + 官网 URL
2. 抓首页链接，让模型挑出适合做知识库的页面
3. 拼成 knowledge base，再回答关于该公司的问题
4. 用缓存避免重复提问时重复打 API

## 和本课概念的对照

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多步 LLM 调用 | 先 `select_relevant_links`，再 `answer_question` |
| JSON 结构化输出 | `response_format={"type": "json_object"}` |
| 流式展示 | `stream=True` + `update_display` |
| OpenRouter 网关 | `base_url=https://openrouter.ai/api/v1` |
| 宣传册 vs 知识库 | 选链接时覆盖 About / Blog / Careers 等更广页面 |

## 怎么跑

1. 准备 `.env`：`OPENROUTER_API_KEY`
2. 同目录需有 `scraper.py`（`fetch_website_links` / `fetch_website_contents`）
3. 从上到下运行；演示格默认抓 `https://huggingface.co`
4. 作者思路细节见其 README（若仓库里有）


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OpenRouter API Key
import os
# 导入标准库 json：把模型返回的 JSON 字符串解析成 Python 字典
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display、流式时 update_display
from IPython.display import Markdown, display, update_display
# 从本地 scraper 模块导入抓取函数：取页面链接列表、取页面正文
from scraper import fetch_website_links, fetch_website_contents
# 从 openai 导入 OpenAI 客户端类：这里会指向 OpenRouter 的兼容端点
from openai import OpenAI


In [ ]:
# ========== 初始化与常量：密钥检查 + 模型名集中管理 ==========

# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)
# OpenRouter 的 OpenAI 兼容 API 基址（字符串影响请求打到哪，勿改语义）
base_url = "https://openrouter.ai/api/v1"
# 从环境变量读取 OpenRouter 密钥
api_key = os.getenv('OPENROUTER_API_KEY')

# 粗检密钥是否存在且足够长（>10），只做友好提示，不中断
if api_key and len(api_key)>10:
    # 看起来像有有效密钥时打印英文提示（影响行为的文案保留原文）
    print("API key looks good so far")
else:
    # 密钥缺失/过短时提示去排查笔记本（文案保留英文）
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# 选链接阶段用的小模型：便宜、够做 JSON 分类
LINKS_MODEL = 'gpt-4o-mini'
# 专家问答阶段用的模型 id（经 OpenRouter 路由）
EXPERT_MODEL = 'gpt-5-nano'


In [ ]:
# ========== 提示模板：链接筛选 + 专家问答（运行时用 .format 填坑） ==========
# 用 {placeholder} 占位，真正调用时再注入 url / company_name / knowledge_base 等

# 系统提示：告诉模型如何从链接列表里挑「知识库」相关页，并只要 JSON
# （prompt 字符串保持英文，影响模型行为，不翻译）
LINKS_SYSTEM_PROMPT = """
You are provided with a list of links found on a webpage.
For a **data bank** (vs. a brochure), we want a **broader** set of links — anything that could inform answers:
- About, Company, Team
- Products, Services, Solutions
- Blog, News, Articles
- Careers, Jobs, Culture
- Contact, Support, FAQ, Help
- Documentation, Docs, Guides

**Exclude:** Terms of Service, Privacy Policy, cookie banners, social media, email `mailto:` links.

Respond in JSON only:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

# 用户提示模板：把具体 URL 与链接列表塞进去，要求返回绝对 URL 的 JSON
LINKS_USER_PROMPT_TEMPLATE = """
Here are the links found on {url}. Select those relevant for a company knowledge base.
Return full absolute URLs in JSON format. Do not include Terms of Service, Privacy, or email links.

Links:
{links}
"""

# 专家系统提示模板：约束「只根据知识库回答」，禁止幻觉与外部知识
EXPERT_SYSTEM_PROMPT_TEMPLATE = """
You are an expert system for {company_name}. Your ONLY source of information is the knowledge base provided below. Answer questions accurately based on it. You must NOT use external knowledge or assumptions.

RULES:
1. Answer questions ONLY based on the provided knowledge base.
2. If the knowledge base does not contain enough information to answer, say: "I don't have enough information in the provided content to answer this question."
3. If the question is unrelated to the company (e.g., general trivia, other companies), say: "This question appears unrelated to {company_name}. I can only answer questions about the information in the knowledge base."
4. When you do have an answer, cite the relevant section or page when helpful (e.g., "According to the About page...").
5. Be concise but accurate. Do not speculate or hallucinate.

OUTPUT FORMAT:
- Do NOT repeat the user's question.
- Start with your answer. If you cite sources, end with a "References:" section listing relevant URLs from the knowledge base.
- Use markdown. Do NOT wrap your response in code blocks.

Example (when you have an answer):
**Answer:** The main product is the Hugging Face Hub and API, which provides access to models, datasets, and ML tools.

**References:**
- https://huggingface.co/about

Example (when you lack information):
I don't have enough information in the provided content to answer this question.
"""

# 专家用户提示模板：知识库正文 + 具体问题
EXPERT_USER_PROMPT_TEMPLATE = """
Knowledge base for {company_name} (from {url}):

---
{knowledge_base}
---

Question: {question}
"""


In [ ]:
# ========== ExpertSystem 类：抓链 → 选链 → 建库 → 流式问答（带缓存） ==========

class ExpertSystem:
    """
    给定公司 URL，构建知识库并回答问题；支持问答缓存以避免重复 API 调用。
    """

    def __init__(self, url, company_name):
        # 保存公司名：后面填进专家 system prompt
        self.company_name = company_name
        # 保存官网入口 URL
        self.url = url
        # 最近一次用户问题（可选状态字段）
        self.user_question = None
        # 创建指向 OpenRouter 的 OpenAI 兼容客户端
        self.client = OpenAI(base_url=base_url, api_key=api_key)
        # 首页上扫到的原始链接（稍后填充）
        self.links = None
        # 模型筛选后的相关链接 JSON
        self.relevant_links = None
        # 链接筛选用的模型名
        self.links_model = LINKS_MODEL
        # 专家问答用的模型名
        self.expert_model = EXPERT_MODEL
        # 拼好的知识库长文本
        self.knowledge_base = None
        # 预留字段（本笔记本后续未再赋值，保持原样）
        self.expert_system = None
        # 问答缓存：规范化后的问题 -> 完整回答文本
        self._cache = {}  # question -> answer (normalized question as key)

        # 抓取首页上的链接列表
        self.links = fetch_website_links(self.url)

        # 让 LLM 从链接里挑出适合进知识库的子集
        self.relevant_links = self.select_relevant_links()

        # 抓各页正文拼知识库，并截断到前 10000 字符控制上下文长度/费用
        self.knowledge_base = self.build_knowledge_base()[:10_000]

    @staticmethod
    def _normalize_question(question: str) -> str:
        """把问题规范化成缓存键：去首尾空白、小写、合并连续空白。"""
        return " ".join(question.strip().lower().split())

    def clear_cache(self):
        """清空问答缓存。"""
        self._cache.clear()

    def _build_prompt(self, template: str, **kwargs) -> str:
        """用关键字参数填充提示模板（str.format）。"""
        return template.format(**kwargs)

    def query_model(self, model, messages, **kwargs):
        # 统一封装 chat.completions.create；额外参数（如 stream / response_format）经 **kwargs 透传
        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            **kwargs
        )
        # 流式时直接返回 stream 迭代器，由调用方消费
        if kwargs.get('stream', False):
            return response
        # 非流式：取出第一条 choice 的文本内容
        return response.choices[0].message.content

    def select_relevant_links(self):
        # 把当前 url 与链接列表填进用户提示
        user_prompt = self._build_prompt(
            LINKS_USER_PROMPT_TEMPLATE,
            url=self.url,
            links="\n".join(str(link) for link in self.links)
        )
        # system 定规则，user 给具体链接
        messages = [
            {"role": "system", "content": LINKS_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
        # 要求 JSON object，再 json.loads 成字典
        links = json.loads(
            self.query_model(self.links_model, messages, response_format={"type": "json_object"})
        )
        # 打印找到多少条相关链接（英文文案保留）
        print(f"Found {len(links['links'])} relevant links")
        return links

    def build_knowledge_base(self):
        # 先抓落地页正文
        contents = fetch_website_contents(self.url)
        # 用 Markdown 标题组织：Landing Page + Relevant Links
        result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
        # 逐个相关链接抓正文并追加
        for link in self.relevant_links['links']:
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_website_contents(link["url"])
        return result

    def _get_expert_messages(self, question: str):
        """组装专家问答的 messages 列表。"""
        # 填公司名到专家 system prompt
        system_prompt = self._build_prompt(
            EXPERT_SYSTEM_PROMPT_TEMPLATE,
            company_name=self.company_name
        )
        # 填公司名、URL、知识库、问题到 user prompt
        user_prompt = self._build_prompt(
            EXPERT_USER_PROMPT_TEMPLATE,
            company_name=self.company_name,
            url=self.url,
            knowledge_base=self.knowledge_base,
            question=question
        )
        return [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

    def _stream_response(self, stream) -> str:
        """边收流式 chunk 边刷新 Markdown 显示，并返回完整文本。"""
        # 累积完整回答
        full_response = ""
        # 创建一个可更新的空 Markdown 显示句柄
        display_handle = display(Markdown(""), display_id=True)
        # 迭代每个流式 chunk
        for chunk in stream:
            # delta.content 可能为 None，用空串兜底
            content = chunk.choices[0].delta.content or ''
            full_response += content
            # 用同一 display_id 原地更新，实现打字机效果
            update_display(Markdown(full_response), display_id=display_handle.display_id)
        return full_response

    def answer_question(self, question: str, use_cache: bool = True):
        """
        回答关于公司的问题；use_cache=True 时先查缓存。
        """
        # 记录当前问题
        self.user_question = question
        # 生成缓存键
        cache_key = self._normalize_question(question)

        # 命中缓存：提示并直接展示，不再打 API
        if use_cache and cache_key in self._cache:
            print("(from cache)")
            display(Markdown(self._cache[cache_key]))
            return

        # 未命中：组 messages，流式调用专家模型
        messages = self._get_expert_messages(question)
        stream = self.query_model(self.expert_model, messages, stream=True)
        # 流式展示并拿到完整文本
        full_response = self._stream_response(stream)
        # 写入缓存供下次复用
        self._cache[cache_key] = full_response


In [ ]:
# ========== 演示：建库 → 提问 → 缓存命中 → 换问题 ==========

# 以 Hugging Face 官网为示例初始化专家系统（会抓链、选链、建库，较耗时）
expert_system = ExpertSystem(url="https://huggingface.co", company_name="Hugging Face")

# 第一次提问：会真正调用专家模型 API（问题字符串保持英文）
expert_system.answer_question("What is the main product of Hugging Face?")

# 相同问题再问一次：应打印 (from cache) 且不再打 API
expert_system.answer_question("What is the main product of Hugging Face?")

# 换一个不同问题：再次走 API
expert_system.answer_question("Does Hugging Face have a careers page?")

# 绕过缓存的用法示例（注释掉，需要时取消注释）：
# expert_system.answer_question("...", use_cache=False)
# 清空缓存：expert_system.clear_cache()
